GenZ Q&A

Prompt [ANS] + enc.encode_ordinary(genz_slang) + [EOT]

In [8]:
import tiktoken
from datasets import load_dataset
import numpy as np

In [4]:

base = tiktoken.get_encoding("gpt2")
enc = tiktoken.Encoding(
    name="gpt2_answer",
    pat_str=base._pat_str,
    mergeable_ranks=base._mergeable_ranks,
    special_tokens={**base._special_tokens, "<answer>": base.n_vocab},
)
ANS, EOT = enc.encode_single_token("<answer>"), enc.eot_token


In [5]:

ds = load_dataset("aiml8726737/gen-alpha-brainrot", num_proc=8)

ids = []
for row in ds["train"]:
	prompt = row["messages"][0]["content"]
	genz_answer = row["messages"][1]["content"]
	curr_ids = enc.encode_ordinary(prompt) + [ANS] + enc.encode_ordinary(genz_answer) + [EOT]
	ids.append(curr_ids)

In [6]:
ds = load_dataset("synk/genz-slang-completions",  num_proc=8)

for row in ds["train"]:
	curr_ids = enc.encode_ordinary(row["prompt"]) + [ANS] + enc.encode_ordinary(row["completion"]) + [EOT]
	ids.append(curr_ids)

In [9]:
arr = np.array(ids, dtype=object)
ixs = np.random.permutation(len(arr))
flat = np.concatenate(arr[ixs]).astype(np.uint16)
print(len(flat))
flat.tofile("genz_qna_finetune.bin")

59165


In [10]:
import os
file = f"genz_qna_finetune.bin"
n = os.path.getsize(file) // 2
print(f"{file}: {n:,} tokens, {n/1e6:.2f}M tokens")

genz_qna_finetune.bin: 59,165 tokens, 0.06M tokens
